# Olasılıksal Dil Modelleri

- Dilin yapısını ve düzenini anlamak ve modellemek için kullanılan istatistiksel yöntemlerdir.

Table of Contents

- [1. N-Gram Modelleri](#1.-N-Gram-Modelleri)
- [2. Gizli Markov Modelleri (Hidden Markov Models)](#2.-Gizli-Markov-Modelleri-(Hidden-Markov-Models):)
    - [2.a. Part of Speech (PoS)](#2.a.-Part-of-Speech-(PoS))

## 1. N-Gram Modelleri

- Bir dizideki (genellikle bir cümledeki) ardışık kelime veya karakter gruplarının olasılıklarını tahmin eder.
- *n* sayısı, gruptaki öğe sayısını belirtir.
    - <b><em>Uni-Gram (n = 1):</em></b> `This` `Is` `Big` `Data` `AI` `Book`
    - <b><em>Bi-Gram (n = 2):</em></b> `This is` `Is Big` `Big data` `Data AI` `AI Book`
    - <b><em>Tri-Gram (n = 3):</em></b> `This is big` `Is big data` `Big data AI` `Data AI Book`

- Bir cümlede bir kelimenin hangi olasılıkla diğer bir kelimeden sonra geleceğini tahmin etmek için kullanılır.
- Hesaplaması basit ve hızlıdır, yerel bağıntıları iyi yakalar
- Bağlam sınırlaması vardır, uzun mesafeleri bağımlılıkları yakalayamazlar. Yüksek veri gereksinimi vardır.

In [1]:
import pandas as pd
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize

from collections import Counter

In [2]:
imdb_review_df = pd.read_csv("IMDB Dataset.csv")

In [59]:
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup as bs
stop_words_eng = set(stopwords.words("english"))

def clean_text(text):
    # Text'in küçük harflere çevirilmesi
    text = text.lower()

    # Text temizliği:
    # HTML tag'leri:
    text = bs(text, "html.parser").get_text()
    # Harf ya da boşluk olmayan tüm karakterler:
    text = re.sub(r"[^A-Za-z\s]+", "", text)
    # Birden fazla tekrar eden boşluk karakterlerinin teke düşürülmesi:
    text = re.sub(r"\s{2, }", " ", text)
    # Birleşik yazılan stop-word'lerin ayırılması:
    text = re.sub(r"cannot", "can not", text)

    # Stop-word'lerin temizlenmesi
    word_list = text.split()
    text = " ".join([word for word in word_list if word not in stop_words_eng])

    return text

In [60]:
reviews = imdb_review_df["review"].iloc[:3000].apply(clean_text)

In [61]:
tokens = [word_tokenize(sentence) for sentence in reviews]

In [62]:
bigrams = []
for token_list in tokens:
    bigrams.extend(list(ngrams(token_list, 2)))

In [63]:
bigrams_freq = Counter(bigrams)

In [75]:
bigrams_freq_df = pd.DataFrame({"Words": bigrams_freq.keys(), "Count": bigrams_freq.values()})
bigrams_freq_df.sort_values(by="Count", ascending=False).head(5)

,Words,Count
1231,"(ever, seen)",131
733,"(ive, seen)",129
9756,"(dont, know)",125
1333,"(special, effects)",124
3080,"(even, though)",120


In [76]:
trigrams = []
for token_list in tokens:
    trigrams.extend(list(ngrams(token_list, 3)))

trigrams_freq = Counter(trigrams)
trigrams_freq_df = pd.DataFrame({"Words": trigrams_freq.keys(), "Count": trigrams_freq.values()})
trigrams_freq_df.sort_values(by="Count", ascending=False).head(5)

,Words,Count
1230,"(ive, ever, seen)",57
9963,"(worst, movie, ever)",28
11828,"(movie, ive, ever)",18
9964,"(movie, ever, seen)",18
9796,"(one, worst, movies)",17


In [82]:
# "En iyi film" ve "En kötü film" olasılıkları:
bme_prob = trigrams_freq[("best", "movie", "ever")]/bigrams_freq[("movie", "ever")]
wme_prob = trigrams_freq[("worst", "movie", "ever")]/bigrams_freq[("movie", "ever")]

print(f"'____ movie ever'daki boşluğa 'best' ya da 'worst' gelme olasılıkları\n'best': {bme_prob*100}\n'worst': {wme_prob*100}")

'____ movie ever'daki boşluğa 'best' ya da 'worst' gelme olasılıkları
'best': 6.451612903225806
'worst': 45.16129032258064


## 2. Gizli Markov Modelleri (Hidden Markov Models):

- Bir dizi gözlemin (kelimeler ya da karakterler gibi) arkasında gizli bir durum dizisinin (örneğin dil bilgisel kategoriler) olduğu varsayımına dayanır.
- Her bir durmun belirli bir olasılıkla başka bir duruma geçeceğini ve her durumun belirli bir gözlemi üreteceğini varsayar.

![Hidden Markov Models Image](https://i.ytimg.com/vi/RWkHJnFj5rY/hq720.jpg?sqp=-oaymwEhCK4FEIIDSFryq4qpAxMIARUAAAAAGAElAADIQj0AgKJD&rs=AOn4CLB1YKn42svs95LecSRssvsHe1dN_w)

- **Kullanım:**
    - Konuşma tanıma
    - Dil modelleme
    - Parça etiketleme (Part-of-Speech tagging)
- **Avantajları:**
    - Bağlam modelleme
    - Verimli algoritmalar
- **Dezavantajları:**
    - Basitleştirici varsayımlar, bağımlılıkların yalnızca bir önceki durumdan geldiğini varsayar
    - Eğitim zorluğu, yüksek miktarda veriye ihtiyaç

### 2.a. Part-of-Speech (PoS)

In [1]:
import pandas as pd
import nltk
from nltk.tag import hmm

train_df = pd.read_csv("words_pos.csv")
train_df.drop(["Unnamed: 0"], axis=1, inplace=True)

In [17]:
# HMM requires tagged data that is a part of a sentence. For the dataset I have it's too over-engineered and does not yield satisfactory results.
# train_data = []
# for i in range(0, 3701):
#     train_data.append(list(zip(train_df["word"].iloc[i*100:(i+1)*100], train_df["pos_tag"].iloc[i*100:(i+1)*100])))


In [18]:
len(train_data)

3701

In [19]:
trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train(train_data)

In [20]:
test_sentence = "i am an successful student".split(" ")
tags = hmm_tagger.tag(test_sentence)

In [21]:
tags

[('i', 'NN'),
 ('am', 'VBP'),
 ('an', 'NN'),
 ('successful', 'NN'),
 ('student', 'NN')]